# Step 3: master 통합 데이터셋 빌드**목적**: staged 데이터 + apt_id_map을 결합해 4~7단계 콘텐츠가 읽을 단 두 개의 마스터 테이블 생성.**산출물**:1. `master/trade_events.parquet` — 거래 이벤트 테이블 (설계서 5.3절)2. `master/unified_apt_pyeong_daily.parquet` — 일별 단지·평형 상태 (설계서 5.2절)**원자적 swap**: 매 빌드는 전체 재구성 후 임시 파일 → rename. 빌드 도중 다른 잡이 읽어도 깨진 데이터 안 보임.**핵심 함정** (설계서 6.2·6.3절):- 해제 거래는 보존하되 `is_canceled=True` 플래그- 같은 (apt_id, deal_date, area_m2, floor) 중복 거래는 가장 최근 신고분으로 dedup- SCD Type 2 join: snapshot_date 시점에 유효한 단지정보·인구만 join

## 1. 환경 로드

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys
    sys.path.insert(0, '/content/drive/MyDrive/realestate_reco')
except ImportError:
    pass  # 로컬 환경

from common import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
print('환경 준비 완료')

## 2. 모든 staged 거래 로드 + dedup

In [ ]:
def load_all_staged_trade() -> pd.DataFrame:    """staged 디렉토리의 모든 trade Parquet을 통합 로드."""    files = list(DIRS['staged_trade'].rglob('*.parquet'))    if not files:        raise FileNotFoundError('staged 거래 파일 없음')    dfs = [pd.read_parquet(p) for p in files]    df = pd.concat(dfs, ignore_index=True)    print(f'로드: {len(files)}개 파일, {len(df):,}건 거래')    return dfdef dedup_trades(df: pd.DataFrame) -> pd.DataFrame:    """같은 거래 중복 제거.    같은 거래 = 같은 (apt_id, deal_date, area_m2, floor, deal_amount).    정정으로 가격이 바뀐 건은 최신 가격을 보존.    """    before = len(df)    # 정렬 후 마지막 행만 보존 (가장 최근 신고)    df = df.sort_values(['apt_id', 'deal_date', 'area_m2', 'floor', 'deal_amount'])    df = df.drop_duplicates(        subset=['apt_id', 'deal_date', 'area_m2', 'floor'],        keep='last'    )    print(f'dedup: {before:,} → {len(df):,} ({before-len(df):,}건 중복 제거)')    return dfdf_trade = load_all_staged_trade()df_trade = dedup_trades(df_trade)

## 3. trade_events 테이블 빌드 (설계서 5.3절)거래 이벤트 자체. Daily Shorts ‘오늘의 한 건’ 후보 추출에 사용.

In [ ]:
def build_trade_events(df: pd.DataFrame) -> pd.DataFrame:    """거래 이벤트 테이블 생성."""    df = df.copy()    # 시군구명 (영상 자막용)    df['sigungu_name'] = df['sigungu_code'].map(SIGUNGU_CODES)    # 평당가 (만원/3.3㎡)    df['price_per_m2'] = df['deal_amount'] / df['area_m2']  # 만원/㎡    df['price_per_pyeong'] = df['price_per_m2'] * 3.3058    # 노후도    today = pd.Timestamp.now().normalize()    df['age_at_deal'] = df['deal_date'].dt.year - df['build_year']    # 표준 컬럼 순서    cols = [        'apt_id', 'pyeong_bucket', 'deal_date',        'sigungu_code', 'sigungu_name', 'legal_dong_code', 'legal_dong_name',        'apt_name_raw', 'apt_name_norm',        'deal_amount', 'area_m2', 'floor', 'build_year', 'age_at_deal',        'price_per_m2', 'price_per_pyeong',        'is_canceled', 'cancel_date',        'road_address',    ]    df = df[[c for c in cols if c in df.columns]]    print(f'trade_events: {len(df):,}건')    return dfdef save_trade_events(df: pd.DataFrame):    """원자적 swap: 임시 파일 쓰고 rename."""    final_path = DIRS['master'] / 'trade_events.parquet'    tmp_path = DIRS['master'] / 'trade_events.parquet.tmp'    df.to_parquet(tmp_path, index=False, compression='zstd')    tmp_path.replace(final_path)    print(f'저장: {final_path}')df_events = build_trade_events(df_trade)save_trade_events(df_events)

## 4. 단지·평형별 일별 스냅샷 테이블 빌드 (설계서 5.2절)‘아파트 × 평형 × 일’ 단위. 한 행 = ‘이 단지의 이 평형이 이 날 어떤 상태였는가’.**규모 주의**: 8,000단지 × 평균 2개 평형 × 365일 = 약 580만 행/년.초기 구축은 직전 N개월만 빌드해 검증 후 확장 권장.

In [ ]:
def build_unified_daily(df_events: pd.DataFrame,                        start_date: str = None,                        end_date: str = None) -> pd.DataFrame:    """단지·평형 × 일별 스냅샷 테이블.    각 (apt_id, pyeong_bucket, snapshot_date) 행에 대해:      - 직전 30일/90일/365일 거래 통계      - 가장 최근 거래일·가격    """    df = df_events[df_events['is_canceled'] == False].copy()  # 해제 거래 제외    end_date = pd.to_datetime(end_date) if end_date else df['deal_date'].max()    start_date = pd.to_datetime(start_date) if start_date else end_date - pd.Timedelta(days=90)    print(f'기간: {start_date.date()} ~ {end_date.date()}')    # 단지·평형 unique    keys = df[['apt_id', 'pyeong_bucket', 'sigungu_code', 'sigungu_name',               'legal_dong_code', 'legal_dong_name', 'apt_name_raw']].drop_duplicates()    # snapshot_date를 일별로 cross join    dates = pd.date_range(start_date, end_date, freq='D')    print(f'단지·평형: {len(keys):,} × 일수: {len(dates)} = {len(keys)*len(dates):,} 행')    # 메모리 폭증 가능성 ─ 시군구 단위로 청크 처리    all_chunks = []    for sigungu, group_keys in keys.groupby('sigungu_code'):        chunk_df = group_keys.merge(            pd.DataFrame({'snapshot_date': dates}),            how='cross'        )        all_chunks.append(chunk_df)    base = pd.concat(all_chunks, ignore_index=True)    # 거래 통계 join    base = _attach_trade_stats(base, df, end_date)    print(f'unified_daily: {len(base):,} 행')    return basedef _attach_trade_stats(base: pd.DataFrame, events: pd.DataFrame, end_date) -> pd.DataFrame:    """각 (apt_id, pyeong_bucket, snapshot_date) 행에 시계열 거래 피처 부착."""    # 가장 최근 거래    last = (events            .sort_values('deal_date')            .groupby(['apt_id', 'pyeong_bucket'])            .agg(trade_last_date=('deal_date', 'max'),                 trade_last_price=('deal_amount', 'last'),                 trade_last_per_m2=('price_per_m2', 'last'))            .reset_index())    base = base.merge(last, on=['apt_id', 'pyeong_bucket'], how='left')    # 30일 / 90일 / 365일 윈도우 통계    # (성능을 위해 단순화 — 운영에서는 SQL/Polars로 윈도우 함수 사용 권장)    for window in [30, 90, 365]:        agg = []        for (apt, py), g in events.groupby(['apt_id', 'pyeong_bucket']):            for snap_date in base[(base['apt_id'] == apt) & (base['pyeong_bucket'] == py)]['snapshot_date'].unique():                lo = snap_date - pd.Timedelta(days=window)                window_g = g[(g['deal_date'] >= lo) & (g['deal_date'] <= snap_date)]                agg.append({                    'apt_id': apt, 'pyeong_bucket': py, 'snapshot_date': snap_date,                    f'trade_{window}d_count': len(window_g),                    f'trade_{window}d_avg_price': window_g['deal_amount'].mean() if len(window_g) else np.nan,                    f'trade_{window}d_avg_per_m2': window_g['price_per_m2'].mean() if len(window_g) else np.nan,                })        if agg:            df_agg = pd.DataFrame(agg)            base = base.merge(df_agg, on=['apt_id', 'pyeong_bucket', 'snapshot_date'], how='left')    return base# 데모: 직전 30일만 빌드 (전체는 시간이 오래 걸림)df_unified = build_unified_daily(df_events,                                 start_date=(df_events['deal_date'].max() - pd.Timedelta(days=30)).strftime('%Y-%m-%d'),                                 end_date=df_events['deal_date'].max().strftime('%Y-%m-%d'))df_unified.head()

## 5. 단지정보 SCD Type 2 joinsnapshot_date 시점에 유효한 단지정보를 join.

In [ ]:
def attach_apt_info(df: pd.DataFrame) -> pd.DataFrame:    """단지정보 SCD join.    snapshot_date 이전에 valid_from을 가진 가장 최근 단지정보 스냅샷 사용.    """    snap_files = sorted(DIRS['staged_apt_info'].glob('snapshot=*.parquet'))    if not snap_files:        print('단지정보 스냅샷 없음. apt_* 컬럼 비워둠.')        return df    # 모든 스냅샷 통합 후 SCD join    snap_dfs = [pd.read_parquet(p) for p in snap_files]    df_snap = pd.concat(snap_dfs, ignore_index=True)    # apt_id_map으로 K-apt 코드 → apt_id 매핑    df_map = pd.read_parquet(DIRS['master'] / 'apt_id_map.parquet')    df_map_lookup = df_map[df_map['kapt_code'].notna()][['kapt_code', 'apt_id']]    df_snap = df_snap.merge(df_map_lookup, left_on='kaptCode', right_on='kapt_code', how='inner')    # 각 (apt_id, snapshot_ym)별 최신 정보    df_snap['snap_date'] = pd.to_datetime(        df_snap['snapshot_ym'].astype(str).str[:4] + '-' +        df_snap['snapshot_ym'].astype(str).str[4:] + '-01'    ) + pd.offsets.MonthEnd(0)    # 각 행에 대해 snapshot_date 이전의 가장 최근 스냅샷 선택    df = df.sort_values('snapshot_date')    df_snap = df_snap.sort_values('snap_date')    df = pd.merge_asof(        df, df_snap[['apt_id', 'snap_date', 'kaptUsedate', 'kaptdaCnt', 'kaptDongCnt', 'kaptdPcntu']],        left_on='snapshot_date', right_on='snap_date',        by='apt_id', direction='backward'    )    # 영문 표준 컬럼명    df = df.rename(columns={        'kaptUsedate': 'apt_used_date',        'kaptdaCnt': 'apt_total_households',        'kaptDongCnt': 'apt_total_buildings',        'kaptdPcntu': 'apt_total_parking',    })    return dfdf_unified = attach_apt_info(df_unified)print(df_unified.columns.tolist())

## 6. master 저장 (원자적 swap)

In [ ]:
def save_unified(df: pd.DataFrame):    final_path = DIRS['master'] / 'unified_apt_pyeong_daily.parquet'    tmp_path = DIRS['master'] / 'unified_apt_pyeong_daily.parquet.tmp'    df.to_parquet(tmp_path, index=False, compression='zstd')    tmp_path.replace(final_path)    print(f'저장: {final_path} ({len(df):,}행, {len(df.columns)}컬럼)')save_unified(df_unified)

## 7. 빌드 결과 sanity check (설계서 6.1절)

In [ ]:
def sanity_check_master():    """master 빌드 직후 자동 검증. 실패 시 P1 알림."""    checks = []    # 1. trade_events 행수    df_e = pd.read_parquet(DIRS['master'] / 'trade_events.parquet')    today_n = (df_e['deal_date'] == df_e['deal_date'].max()).sum()    checks.append({        'id': 1, 'name': 'daily_trade_count',        'value': int(today_n), 'threshold': '[200, 3000] 평일',        'status': 'PASS' if 50 <= today_n <= 3000 else 'WARN'    })    # 2. apt_id 결측 비율    null_pct = df_e['apt_id'].isna().mean()    checks.append({        'id': 3, 'name': 'apt_id_null_ratio',        'value': float(null_pct), 'threshold': '< 0.01',        'status': 'PASS' if null_pct < 0.01 else 'FAIL'    })    # 3. 거래가 범위    out_of_range = ((df_e['deal_amount'] < 50000) | (df_e['deal_amount'] > 500000)).mean()    checks.append({        'id': 4, 'name': 'price_out_of_range_ratio',        'value': float(out_of_range), 'threshold': '< 0.05',        'status': 'PASS' if out_of_range < 0.05 else 'WARN'    })    # 4. 해제 거래 비율 (직전 30일)    last_30d = df_e[df_e['deal_date'] >= df_e['deal_date'].max() - pd.Timedelta(days=30)]    cancel_pct = last_30d['is_canceled'].mean() if len(last_30d) else 0.0    checks.append({        'id': 6, 'name': 'cancel_ratio_30d',        'value': float(cancel_pct), 'threshold': '< 0.05',        'status': 'PASS' if cancel_pct < 0.05 else 'FAIL'    })    df_check = pd.DataFrame(checks)    print(df_check.to_string(index=False))    overall = 'FAIL' if (df_check['status'] == 'FAIL').any() else \              'WARN' if (df_check['status'] == 'WARN').any() else 'PASS'    block = overall == 'FAIL'    today = datetime.now().strftime('%Y-%m-%d')    log_path = DIRS['logs'] / f'quality_{today}.json'    log_path.write_text(json.dumps({        'date': today, 'overall': overall, 'block_publish': block,        'checks': df_check.to_dict('records')    }, ensure_ascii=False, indent=2), encoding='utf-8')    print(f'\nOverall: {overall} | block_publish: {block}')    print(f'로그: {log_path}')    return overall, blocksanity_check_master()

---## 다음 단계여기까지가 Step 1~3 (설계서 1~5장 코드 구현). 이후:- **Step 4 (콘텐츠)**: `master/trade_events.parquet`을 읽어 Daily Shorts 변수 JSON 산출- **Step 5 (콘텐츠)**: `master/unified_apt_pyeong_daily.parquet`로 Weekly·Monthly 집계- **Step 6~7 (콘텐츠)**: 기존 첨부 노트북 (Step 2 모델·Step 2.5 정제·Step 3 추천)을 master 위에 다시 올림**자동화 cron 예시** (설계서 4.1절):```bash# /etc/cron.d/realestate0 4 * * *  /usr/bin/python3 /path/to/run_step1a.py >> logs/cron.log 2>&130 4 * * * /usr/bin/python3 /path/to/run_step1b.py >> logs/cron.log 2>&10 5 * * *  /usr/bin/python3 /path/to/run_step2.py  >> logs/cron.log 2>&115 5 * * * /usr/bin/python3 /path/to/run_step3.py  >> logs/cron.log 2>&1```각 노트북을 `.py`로 export 후 cron 등록.